In [1]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time

bank_companies = ["PRVU", "SBL", "EBL", "PCBL", "HBL"]


def clean_number(value):
    if value is None:
        return None

    value = value.strip()

    if value in ["", "-", "N/A"]:
        return None

    value = value.replace(",", "")

    try:
        return float(value)
    except:
        return None


def scrape_company(symbol):

    url = f"https://merolagani.com/CompanyDetail.aspx?symbol={symbol}"

    headers = {
        "User-Agent": "Mozilla/5.0"
    }

    try:
        response = requests.get(url, headers=headers)

        if response.status_code != 200:
            print(f"Could not access {symbol}")
            return None

        soup = BeautifulSoup(response.text, "html.parser")
 
        def get_value(label):
            for tag in soup.find_all(["td", "th"]):

                if label.lower() in tag.get_text(strip=True).lower():

                    sibling = tag.find_next_sibling(["td", "th"])

                    if sibling:
                        return sibling.get_text(strip=True)

            return None

        price = get_value("Market Price")
        pe = get_value("P/E Ratio")
        pb = get_value("PBV")

        company_data = {
            "Ticker": symbol,
            "Market Price": clean_number(price),
            "P/E Ratio": clean_number(pe),
            "P/B Ratio": clean_number(pb)
        }

        return company_data

    except Exception as e:
        print("Error:", e)
        return None


all_data = []

for company in bank_companies:

    print("Scraping:", company)

    result = scrape_company(company)

    if result:
        all_data.append(result)

    time.sleep(1)

df = pd.DataFrame(all_data)

print("\nFinal Data:")
print(df)

df.to_csv("bank_data.csv", index=False)


print("\nData saved to bank_data.csv")

Scraping: PRVU
Scraping: SBL
Scraping: EBL
Scraping: PCBL
Scraping: HBL

Final Data:
  Ticker  Market Price  P/E Ratio  P/B Ratio
0   PRVU         187.2      38.84       1.28
1    SBL         387.1      21.97       1.79
2    EBL         694.0      21.45       2.84
3   PCBL         232.4      11.82       1.36
4    HBL         191.1      42.47       1.11

Data saved to bank_data.csv
